# A.6 找不到或文件不可靠時怎麼辦？

# A：上下文學習與 RAG

前置：第7章prompt與對話。不需要圖片或MoE。把模型想成可以讀你遞給它的小抄的答題者：先手動給例子，再從本地文件找相關段落。檢索成功不等於回答一定正確。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像先找小抄，再核對答案來源**

找到對文件與用對資料，是兩個不同分數。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rag.svg")))

**先想一想：**回傳一段文件，就一定足夠作答嗎？

找不到證據時，承認缺資料；無關文件不能被當成答案。文件裏還有可能帶幹擾指令，沿用9.7的任務/資料信任邊界。

**把直覺寫成數學：**低score、空hits、衝突sources應另有標籤與fallback，不用閾值魔法保證誠實。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.retrieval import retrieve

docs = [{"id": "d1", "text": "貓在窗邊"}]
hits = retrieve("書店地址", docs)
print(hits, "資料不足，應請求來源或說明無法確定" if not hits else "需覈對證據")
assert hits == []

**如何讀結果：**規則fallback在檢索器層明確執行；LLM本身是否誠實仍是另一項測試。

**只改一件事：**只加入一個有關鍵字但地址缺失的文件，指出非空hits仍不足。
